# MCP Tutorial - Exploring MCP Server Functions

In this notebook, you'll get a hands-on introduction to the **Model Context Protocol (MCP)** by exploring the functions that power an IT support system. We'll import and call server functions directly to understand what data is available and how each server responds - including how errors are designed to help AI models recover gracefully.

**What you'll learn:**

- What MCP is and why it's useful for building AI-powered applications
- How 5 specialized servers work together to form a complete IT support system
- How each server function behaves with both successful queries and error responses
- Why structured error messages are the key to reliable AI tool use

# What is MCP?

**Model Context Protocol (MCP)** is a standardized protocol for connecting AI models with external tools and data sources. Think of it as a universal adapter between an AI model and the services it needs to interact with.

## Why do we need it?

When building AI-powered applications, we often want the model to interact with external systems like databases, APIs, or file systems. Without a standard protocol, every integration requires custom code. MCP solves this by providing:

- **Tool discovery** - the AI model can ask "what tools are available?" and get a structured list with descriptions and parameter schemas
- **Standardized communication** - every tool follows the same request/response format, regardless of what it does internally
- **Server isolation** - each service runs as an independent process, so failures in one server don't affect others

## Our IT Support System

In this tutorial, we built a complete IT support system using 5 specialized MCP servers. Each server handles a different part of the business:

```
                         ┌──────────────────┐
                         │    AI Model      │
                         │   (gpt-5-nano)   │
                         └────────┬─────────┘
                                  │
                         ┌────────┴─────────┐
                         │  MCP Orchestrator │
                         └────────┬─────────┘
              ┌─────────┬────────┼────────┬─────────┐
              ▼         ▼        ▼        ▼         ▼
         ┌────────┐┌────────┐┌────────┐┌────────┐┌────────┐
         │ Ticket ││Customer││Billing ││  KB    ││ Asset  │
         │ Server ││ Server ││ Server ││ Server ││ Server │
         └────────┘└────────┘└────────┘└────────┘└────────┘
```

- **Ticket Server** - search and manage support tickets, view metrics, find similar tickets
- **Customer Server** - look up customer information, check status, get SLA terms
- **Billing Server** - view invoices, check payment status, calculate outstanding balances
- **Knowledge Base Server** - search for articles and solutions to technical problems
- **Asset Server** - track hardware and software assets, check warranty status

## How we'll explore MCP in this notebook

All server functions in this tutorial are written as **regular Python functions** that can be imported and called directly. In this notebook, we'll start by exploring each server this way - importing its functions and calling them to understand the data structures and error handling patterns.

At the end, we'll look at how the full MCP protocol ties everything together with an AI model.

# Setup

Before running this notebook, make sure the server files are accessible and dependencies are installed.

## Upload Server Files

This notebook imports functions directly from the MCP server files. At the end of the notebook, we'll also use `interactive_client.py` to run the full MCP system from a terminal. Depending on your environment:

**Google Colab:** Upload these 7 Python files using the Files panel on the left sidebar:

1. `ticket_server.py` - Support ticket management
2. `customer_server.py` - Customer database and SLA lookup
3. `billing_server.py` - Invoices and payment tracking
4. `kb_server.py` - Knowledge base search
5. `asset_server.py` - Hardware/software asset tracking
6. `mcp_client.py` - MCP orchestrator (coordinates all servers)
7. `interactive_client.py` - Command-line chat interface

**Local Jupyter:** Make sure all files are in the same directory as this notebook (they should be if you cloned the repository).

## Install Dependencies

We only need the MCP SDK and a few supporting packages. The server functions themselves are pure Python with no additional dependencies.

In [ ]:
!pip install -q mcp==1.27.0 nest-asyncio==1.6.0 typing-extensions==4.13.2 2>/dev/null

print("✅ Dependencies installed!")

# Exploring the Server Functions

Now that we have everything set up, let's explore each server's functions. We'll import them directly and call them as regular Python functions.

For each server, we'll look at two things:

- **Successful queries** - what the data looks like when the requested information exists
- **Error responses** - what happens when data doesn't exist, and how the error messages are structured to help the AI model recover

Pay attention to the error responses. They include fields like `suggested_actions` and `follow_up_tools` that tell the AI model what to try next. This is what makes MCP-based systems more reliable than simple API calls.

## Ticket Server

The ticket server manages support tickets for IT issues like Windows BSOD errors, Linux permission problems, and macOS crashes. It provides functions for searching tickets, getting details, viewing metrics, and finding similar tickets.

Let's start by searching for critical priority tickets.

In [ ]:
# Import ticket server functions
from ticket_server import search_tickets, get_ticket_details, get_ticket_metrics, find_similar_tickets_to, TICKETS

# Example 1: Search for critical priority tickets (SUCCESS)
print("=" * 60)
print("Example 1: Searching for critical tickets")
print("=" * 60)
critical_tickets = search_tickets(priority="critical")
print(f"Found {critical_tickets['total_count']} critical tickets:\n")
for ticket in critical_tickets['tickets']:
    print(f"  {ticket['ticket_id']}: {ticket['subject']}")

Now let's look at how the server handles requests for specific tickets. We'll fetch an existing ticket to see the full data structure, and then try to get one that doesn't exist to see the error response with recovery hints.

In [ ]:
# Example 2: Get details of an existing ticket (SUCCESS)
print("\n" + "=" * 60)
print("Example 2: Getting details for existing ticket TKT-1001")
print("=" * 60)
ticket = get_ticket_details("TKT-1001")
print(f"Ticket: {ticket['ticket_id']}")
print(f"Subject: {ticket['subject']}")
print(f"Status: {ticket['status']}")
print(f"Priority: {ticket['priority']}")
print(f"Description: {ticket['description'][:100]}...")

# Example 3: Try to get a non-existent ticket (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 3: Attempting to get non-existent ticket TKT-9999")
print("=" * 60)
error_response = get_ticket_details("TKT-9999")
print("This ticket doesn't exist. Here's the error response with LLM hints:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Notice the 'suggested_actions' and 'follow_up_tools' that guide the LLM!")

The ticket server also provides aggregate metrics that give an overview of the support team's workload over a given time period.

In [ ]:
# Example 4: Get ticket metrics (SUCCESS)
print("\n" + "=" * 60)
print("Example 4: Getting ticket metrics for last 7 days")
print("=" * 60)
metrics = get_ticket_metrics("last_7_days")
print(f"Ticket Metrics (Last 7 Days):")
print(f"  Total: {metrics['total_tickets']}")
print(f"  Open: {metrics['open_tickets']}")
print(f"  In Progress: {metrics['in_progress_tickets']}")
print(f"  Resolved: {metrics['resolved_tickets']}")
print(f"  Avg Resolution Time: {metrics['avg_resolution_time_hours']} hours")

## Customer Server

The customer server stores information about the companies we support, including their contact details, account status, and SLA (Service Level Agreement) terms. Each customer has a tier (basic, standard, or premium) that determines their response time guarantees.

Let's look up a customer and see what happens when we search for one that doesn't exist.

In [ ]:
# Import customer server functions
from customer_server import lookup_customer, check_customer_status, get_sla_terms, list_customer_contacts

# Example 1: Look up an existing customer (SUCCESS)
print("=" * 60)
print("Example 1: Looking up existing customer CUST-001")
print("=" * 60)
customer = lookup_customer(customer_id="CUST-001")
print(f"Customer: {customer['company_name']}")
print(f"Tier: {customer['tier']}")
print(f"Status: {customer['status']}")
print(f"Account Manager: {customer['account_manager']}")

# Example 2: Try to look up non-existent customer (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to look up non-existent customer CUST-999")
print("=" * 60)
error_response = lookup_customer(customer_id="CUST-999")
print("This customer doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 The LLM can use these hints to try a different approach!")

Let's check the SLA terms for a customer. This information helps the AI model understand response time commitments and prioritize support accordingly.

In [ ]:
# Example 3: Get SLA terms for existing customer (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Getting SLA terms for CUST-001")
print("=" * 60)
sla = get_sla_terms("CUST-001")
print(f"SLA for {sla['company_name']}:")
print(f"  Level: {sla['sla_terms']['level']}")
print(f"  Response Time: {sla['sla_terms']['response_time_hours']} hours")
print(f"  Resolution Time: {sla['sla_terms']['resolution_time_hours']} hours")
print(f"  Support Hours: {sla['sla_terms']['support_hours']}")

## Billing Server

The billing server tracks invoices and payments for each customer. It can retrieve invoices by customer or invoice ID, check payment status, and calculate outstanding balances.

Let's get the invoices for a customer and see how the server handles an invalid invoice ID.

In [ ]:
# Import billing server functions
from billing_server import get_invoice, check_payment_status, calculate_outstanding_balance

# Example 1: Get invoices for an existing customer (SUCCESS)
print("=" * 60)
print("Example 1: Getting invoices for customer CUST-001")
print("=" * 60)
invoices = get_invoice(customer_id="CUST-001")
print(f"Total invoices for customer: {invoices['total_invoices']}\n")
for inv in invoices['invoices'][:3]:  # Show first 3
    print(f"  {inv['invoice_id']}: ${inv['amount']} - {inv['status']}")

# Example 2: Try to get invoice with invalid ID (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to get invoice INV-9999 (doesn't exist)")
print("=" * 60)
error_response = get_invoice(invoice_id="INV-9999")
print("This invoice doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Notice how the error suggests using customer_id instead!")

We can also calculate the total outstanding balance for a customer, which aggregates all unpaid invoices into a single summary.

In [ ]:
# Example 3: Calculate outstanding balance (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Calculating outstanding balance for CUST-002")
print("=" * 60)
balance = calculate_outstanding_balance("CUST-002")
print(f"Outstanding Balance for Customer:")
print(f"  Total: ${balance['outstanding_balance']}")
print(f"  Overdue: ${balance['overdue_amount']}")
print(f"  Unpaid Invoices: {balance['number_of_unpaid_invoices']}")

## Knowledge Base Server

The knowledge base stores technical articles and troubleshooting guides. The AI model uses this server to find solutions for common IT problems. Unlike the other servers, a search with no results returns an empty list rather than an error - the AI model can simply try different search terms.

Let's search for articles about a common issue.

In [ ]:
# Import knowledge base server functions
from kb_server import search_solutions, get_article

# Example 1: Search for BSOD articles (SUCCESS)
print("=" * 60)
print("Example 1: Searching for articles about BSOD")
print("=" * 60)
results = search_solutions("BSOD", limit=3)
print(f"Found {results['total_count']} articles about BSOD:\n")
for article in results['results']:
    print(f"  {article['article_id']}: {article['title']}")
    print(f"    Relevance: {article['relevance_score']}, Views: {article['views']}")

# Example 2: Search with no results (EMPTY RESULT - NOT AN ERROR)
print("\n" + "=" * 60)
print("Example 2: Searching for articles about 'xyz123nonexistent'")
print("=" * 60)
no_results = search_solutions("xyz123nonexistent", limit=3)
print(f"Found {no_results['total_count']} articles.")
print("Note: No error - just empty results. LLM can try different search terms.")

Let's retrieve a full article by its ID and also see what happens when we request one that doesn't exist.

In [ ]:
# Example 3: Get an existing article (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Getting full article KB-001")
print("=" * 60)
article = get_article("KB-001")
print(f"Article: {article['title']}")
print(f"Category: {article['category']}")
print(f"\nContent Preview:")
print(article['content'][:200] + "...")

# Example 4: Try to get non-existent article (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 4: Attempting to get non-existent article KB-999")
print("=" * 60)
error_response = get_article("KB-999")
print("This article doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 The error suggests using search_solutions to find relevant articles!")

## Asset Server

The asset server tracks hardware and software assets including servers, workstations, and laptops. It also monitors warranty status and software licenses, which is important for planning replacements and renewals.

Let's look up an asset and see what information is available.

In [ ]:
# Import asset server functions
from asset_server import lookup_asset, check_warranty

# Example 1: Look up an existing asset (SUCCESS)
print("=" * 60)
print("Example 1: Looking up asset AST-SRV-001")
print("=" * 60)
asset = lookup_asset(asset_id="AST-SRV-001")
print(f"Asset: {asset['hostname']}")
print(f"Type: {asset['asset_type']}")
print(f"Manufacturer: {asset['manufacturer']} {asset['model']}")
print(f"Location: {asset['location']}")

# Example 2: Try to look up non-existent asset (ERROR WITH HINTS)
print("\n" + "=" * 60)
print("Example 2: Attempting to look up non-existent asset AST-999")
print("=" * 60)
error_response = lookup_asset(asset_id="AST-999")
print("This asset doesn't exist. Here's the error response:\n")
import json
print(json.dumps(error_response, indent=2))
print("\n💡 Error provides context and suggests alternative approaches!")

Finally, let's check the warranty status for an asset. This helps the AI model determine whether a device is still covered and when the warranty expires.

In [ ]:
# Example 3: Check warranty for existing asset (SUCCESS)
print("\n" + "=" * 60)
print("Example 3: Checking warranty for asset AST-SRV-001")
print("=" * 60)
warranty = check_warranty("AST-SRV-001")
w = warranty['warranty']
print(f"Warranty for {warranty['hostname']}:")
print(f"  Coverage: {w['coverage_type']}")
print(f"  Status: {w['status']}")
print(f"  End Date: {w['end_date']}")
print(f"  Days Remaining: {w['remaining_days']}")
print(f"  Expired: {w['is_expired']}")

# The Full MCP Protocol in Action

In this notebook, we imported server functions directly as regular Python. This is a great way to explore the data and understand how each server works, but it's **not using the MCP protocol** - we're just calling Python functions.

The real power of MCP comes when you connect everything through the **orchestrator**. Here's what happens in the full system:

```
1. Orchestrator starts each server as a separate subprocess
   (python3 ticket_server.py, python3 customer_server.py, ...)
                              │
2. Orchestrator asks each server: "What tools do you have?"
   Servers respond with tool names, descriptions, and parameter schemas
                              │
3. User asks a question in natural language
   "What are the critical tickets for customer CUST-001?"
                              │
4. Orchestrator sends the question + all 20 tool definitions to gpt-5-nano
                              │
5. gpt-5-nano decides which tools to call:
   → search_tickets(customer_id="CUST-001", priority="critical")
                              │
6. Orchestrator routes the call to the correct server (ticket_server)
   and returns the result back to gpt-5-nano
                              │
7. gpt-5-nano may call more tools or formulate a final answer
```

The communication between the orchestrator and each server happens over **stdio** (stdin/stdout), which is the standard MCP transport. This means the servers run as independent processes - they could even be on different machines.

## Running the Interactive Client

The repository includes `interactive_client.py`, a command-line script that runs the full MCP system. It starts all 5 servers, connects them through the orchestrator, and lets you chat with gpt-5-nano using natural language.

This script needs to run from a **terminal**, not from a notebook cell. Here's how to do it in each environment:

**Google Colab:** Click the Terminal icon in the bottom-left corner to open a terminal on the right side. Then run:

```bash
cd /content
pip install -q mcp==1.27.0 nest-asyncio==1.6.0 openai==2.30.0 2>/dev/null
python interactive_client.py
```

**Local machine:** Open a terminal in the project directory and run:

```bash
python interactive_client.py
```

The script will ask you for your OpenAI API key if it's not already set in the environment.

You can then ask questions like:
- "What are all the critical priority tickets?"
- "Show me customer CUST-001's information and SLA terms"
- "Which customers have both open tickets and overdue invoices?"

gpt-5-nano will automatically discover the 20 available tools, decide which ones to call, and chain multiple calls together to answer complex questions - using the same functions and error responses we explored in this notebook.

# Key Takeaways

Here are the main things to remember from this notebook:

1. **MCP servers are just Python functions underneath** - you can import and call them directly without any protocol infrastructure, which makes testing and debugging straightforward
2. **Error responses are built for AI models, not humans** - they include `suggested_actions`, `follow_up_tools`, and `retryable` flags so the model knows exactly what to try next when something fails
3. **Each server owns its domain** - tickets, customers, billing, knowledge base, and assets are separated into independent servers, but the data is interconnected across them
4. **The MCP protocol adds orchestration on top** - in production, the orchestrator starts servers as subprocesses, discovers their tools automatically, and lets the AI model decide which tools to call based on the user's question

In the next notebook, we'll explore advanced MCP features: **Resources**, **Prompts**, and **Sampling**.